# Example: Use the Long LFMC dataset remotely

This notebook shows how to open the public long LFMC `.zarr` dataset directly from Source Cooperative without downloading the whole thing first.

The dataset is stored remotely, and `xarray` reads only the pieces you ask for. That means opening the dataset is not the same thing as downloading the whole dataset.

## If needed: install packages

If your Python environment does not already have these packages, run the cell below.

In [ ]:
%pip install s3fs zarr

## Import the packages we need

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.path import Path
from pyproj import Transformer

## Define the remote dataset location

This is the public scientific LFMC dataset in its native `EPSG:5070` grid.

In [ ]:
source_endpoint_url = "https://data.source.coop"
source_store = "s3://rseg/long-lfmc-test/lfmc_maps.zarr"

print("Source endpoint:", source_endpoint_url)
print("Remote store:", source_store)

## Open the dataset

This opens the remote store lazily. It reads metadata now, but the actual array values are pulled only when you ask for them. Initially open of the dataset will take about one minute.

In [ ]:
ds = xr.open_zarr(
    source_store,
    chunks={},
    consolidated=True,
    storage_options={
        "anon": True,
        "client_kwargs": {"endpoint_url": source_endpoint_url},
    },
)

ds

## Inspect the dataset

A useful first step is to look at the dimension sizes, variable names, and coordinate names.

In [ ]:
print("Dimensions:")
print(dict(ds.sizes))

print("\nData variables:")
print(list(ds.data_vars))

print("\nCoordinates:")
print(list(ds.coords))

## Look at the time coordinate

This lets you see what dates are available.

In [ ]:
ds.time

## Pull a small time range

Here we request only a few days, which is a small remote read compared with the full dataset.

In [ ]:
time_subset = ds[["lfmc_ens_mean", "lfmc_ens_std"]].sel(time=slice("2023-07-10", "2023-07-15"))
time_subset

## Query one point by latitude and longitude

The dataset grid is in `EPSG:5070`, not latitude/longitude. So first we convert lon/lat into the dataset's `x` and `y` coordinates.

In [ ]:
point_lat = 34.2280
point_lon = -118.8259

transformer = Transformer.from_crs("EPSG:4326", "EPSG:5070", always_xy=True)
point_x, point_y = transformer.transform(point_lon, point_lat)

print("Requested lon/lat:", point_lon, point_lat)
print("Converted x/y:", point_x, point_y)

In [ ]:
point_series = ds["lfmc_ens_mean"].sel(x=point_x, y=point_y, method="nearest")

print("Nearest grid x:", float(point_series["x"].item()))
print("Nearest grid y:", float(point_series["y"].item()))

point_series.to_series().head()

## Plot the LFMC time series for that point

In [ ]:
point_series.plot(figsize=(10, 4))
plt.title("LFMC ensemble mean at one grid cell")
plt.ylabel("LFMC")
plt.show()

## Plot one map slice

This requests one date from the remote dataset and plots it.

In [ ]:
map_slice = ds["lfmc_ens_mean"].sel(time="2023-07-15")
map_slice

In [ ]:
map_slice.plot(figsize=(8, 6), robust=True)
plt.title("LFMC ensemble mean on 2023-07-15")
plt.show()

## Select an area with a polygon

In this example, we define a polygon in longitude/latitude, convert it to the dataset grid, and then use it to select part of one LFMC map.

To keep the remote read smaller, we first subset to the polygon's bounding box and then apply the polygon mask inside that smaller area.

In [ ]:
polygon_lon_lat = [
    (-123.0, 37.0),
    (-122.0, 36.8),
    (-121.6, 37.5),
    (-122.1, 38.1),
    (-122.9, 37.8),
]

polygon_xy = np.array([transformer.transform(lon, lat) for lon, lat in polygon_lon_lat])
polygon_xy

In [ ]:
polygon_x = polygon_xy[:, 0]
polygon_y = polygon_xy[:, 1]

x_min = polygon_x.min()
x_max = polygon_x.max()
y_min = polygon_y.min()
y_max = polygon_y.max()

polygon_subset = ds["lfmc_ens_mean"].sel(
    time="2023-07-15",
    x=slice(x_min, x_max),
    y=slice(y_min, y_max),
)

polygon_subset

In [ ]:
xx, yy = np.meshgrid(polygon_subset["x"].values, polygon_subset["y"].values)
grid_points = np.column_stack([xx.ravel(), yy.ravel()])

polygon_path = Path(polygon_xy)
inside_polygon = polygon_path.contains_points(grid_points).reshape(xx.shape)

polygon_mask = xr.DataArray(
    inside_polygon,
    coords={"y": polygon_subset["y"], "x": polygon_subset["x"]},
    dims=("y", "x"),
)

polygon_selected = polygon_subset.where(polygon_mask)
polygon_selected

In [ ]:
polygon_selected.plot(figsize=(8, 6), robust=True)
plt.title("LFMC inside a user-defined polygon on 2023-07-15")
plt.show()

## What happened here?

- `xr.open_zarr(...)` opened the remote dataset without downloading the full store.
- Inspecting metadata and coordinates is cheap.
- Asking for a time slice, a point series, one map slice, or one polygon subset triggers remote reads for only those pieces.
- If you need the full dataset on disk, use the separate full-download instructions.